# IRR Irregularity Events

This notebook reads the detected IRR irregularity windows and exports one contextual plot per event.

- Intentional `NaN` gaps are ignored by the detector.
- Each saved plot includes time before and after the event and shades the event window.
- Set `EVENT_LIMIT` or `FILTER_METRICS` if you want a smaller batch.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT_DIR = Path.cwd().resolve().parent if Path.cwd().name == "plots" else Path.cwd().resolve()
if str(ROOT_DIR / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT_DIR / "scripts"))

from stw_irr_irregularity_events import export_event_plots

EVENTS_CSV = ROOT_DIR / "reports" / "stw_irr_irregularity_events.csv"
PARQUET_PATH = ROOT_DIR / "final output" / "stw_mV_Irr.parquet"
PLOTS_DIR = ROOT_DIR / "plots" / "irr_irregularity_event_plots"
PLOT_INDEX_CSV = ROOT_DIR / "reports" / "stw_irr_irregularity_plot_index.csv"
FILTER_METRICS = None
EVENT_LIMIT = None
PLOT_CONTEXT_OVERRIDE = None
DPI = 150


In [ ]:
events = pd.read_csv(
    EVENTS_CSV,
    parse_dates=["event_start", "event_end", "peak_datetime", "plot_start", "plot_end"],
)
if FILTER_METRICS:
    events = events[events["metric"].isin(FILTER_METRICS)].copy()
if EVENT_LIMIT is not None:
    events = events.head(EVENT_LIMIT).copy()

events[["plot_label", "metric", "event_start", "event_end", "rule"]].head(20)


In [ ]:
plot_results = export_event_plots(
    PARQUET_PATH,
    events,
    PLOTS_DIR,
    context_hours_override=PLOT_CONTEXT_OVERRIDE,
    plot_index_output=PLOT_INDEX_CSV,
    dpi=DPI,
)
plot_results[["plot_label", "plot_metric", "plot_filename", "irregularity_start", "irregularity_end"]].head(20)
